# Monte Carlo Control with Exploring Starts — Blackjack

**Reproduces S&B Fig 5.2** (§5.3, p.100).

Finds the optimal policy `pi*` and optimal state-value function
`V*(s) = max_a Q*(s, a)` for Blackjack using **Generalized Policy Iteration**
with Monte Carlo evaluation and greedy improvement.

Each episode does:
1. **Exploring start** — sample `(s_0, a_0)` uniformly at random.
2. **Generate episode** — follow the current greedy policy after step 0.
3. **Policy evaluation** — update `Q(s, a)` via incremental average of returns.
4. **Policy improvement** — `pi(s) <- argmax_a Q(s, a)`, interleaved with (3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rl.envs.blackjack import Blackjack
from rl.monte_carlo.mc_control_es import mc_control_es
from blackjack_plots import plot_fig_5_2

## Run MC ES

We use **1,000,000 episodes** (the book uses 500k; we use more to reduce
boundary-state noise). Takes ~2–5 min.

In [ ]:
env = Blackjack()
rng = np.random.default_rng(seed=0)

n_episodes = 1_000_000
Q, pi = mc_control_es(env, rng, n_episodes=n_episodes)

V_star = Q.max(axis=1)
print(f'Q  -> shape={Q.shape},  range=[{Q.min():+.3f}, {Q.max():+.3f}]')
print(f'pi -> STICK fraction = {(pi[:200] == env.STICK).mean():.1%}')
print(f'V* -> range=[{V_star[:200].min():+.3f}, {V_star[:200].max():+.3f}]')

## Fig 5.2 — Optimal policy and value function

In [ ]:
fig = plot_fig_5_2(Q, pi, n_episodes=n_episodes)
plt.show()

## Optimal policy grid (text)

Cell-by-cell view matching the book's layout. `S` = STICK, `H` = HIT.

In [ ]:
for ace in [1, 0]:
    label = 'USABLE ACE' if ace else 'NO USABLE ACE'
    print(f'\n  pi*  --  {label}')
    print(f'  {"":4s}  ' + '  '.join(f'{d:>2s}' for d in ['A','2','3','4','5','6','7','8','9','10']))
    print('  ' + '-' * 40)
    for ps in range(21, 11, -1):
        row = []
        for ds in range(1, 11):
            s = env._encode(ps, ds, ace)
            row.append(' S' if pi[s] == env.STICK else ' H')
        print(f'  {ps:4d}  ' + '  '.join(row))

## Reading the optimal policy

### Usable ace (top row)

With a usable ace the player has a **safety net**: if a hit pushes the sum
past 21, the ace converts from 11 to 1 and prevents a bust.

| Player sum | Action | Why |
|---|---|---|
| 19–21 | STICK | Hand is already strong; hitting risks losing the ace advantage for minimal gain. |
| 18 | Depends on dealer | Against weak dealers (2–8): STICK. Against strong (A, 9, 10): HIT — matches real basic strategy. |
| 12–17 | HIT | The ace absorbs bust risk, so hitting is safe and might improve a weak hand. |

The STICK region is small — the ace safety net makes aggressive play optimal.

### No usable ace (bottom row)

Without the safety net, every hit risks busting. The policy balances two
competing risks:

| Dealer card | Player threshold | Why |
|---|---|---|
| **Weak (2–6)** | STICK on ~12–13+ | Dealer is likely to bust (must hit to 17 from a low card). "Let the dealer bust." |
| **Strong (7–10, A)** | STICK on 17+ only | Dealer will likely reach 17–21. Player must improve by hitting, even at the risk of busting. |

This creates the characteristic **step** in the policy boundary visible in
the bottom-left panel.

### V\* surfaces (right column)

- **V\* ≈ +1 at sum=21** — almost certain win.
- **V\* decreases with player sum** — weaker hands win less.
- **V\* dips against dealer A and 10** — the dealer's strongest showing cards.
- **Usable-ace V\* > no-ace V\*** — the safety net genuinely improves returns.

### Note on boundary noise

A few cells near the STICK/HIT boundary may differ from the theoretical
optimum (real basic strategy). At these states `Q(s, STICK) ≈ Q(s, HIT)`
(the difference is < 1%), so finite-sample noise can flip the policy.
Running with more episodes (2M+) or a different seed shifts these cells.

## Spot-check V\* at notable states

In [ ]:
checks = [
    (21,  2, True,  'near-certain win'),
    (21, 10, False, 'strong but dealer could tie'),
    (18,  6, True,  'moderate hand, weak dealer'),
    (14,  8, False, 'weak hand, strong dealer'),
    (12, 10, False, 'worst common position'),
]
print(f'{"state":>24s}   {"V*":>6s}   {"pi*":>5s}   note')
print('-' * 70)
for ps, ds, ace, note in checks:
    s = env._encode(ps, ds, ace)
    action = 'STICK' if pi[s] == env.STICK else 'HIT'
    ace_str = 'ace' if ace else 'no-ace'
    print(f'  ({ps:2d}, dealer={ds:2d}, {ace_str:6s})   {V_star[s]:+.3f}   {action:>5s}   {note}')